In [7]:
# Load environment variables and verify the project setup.
import sys
from pathlib import Path

# Find the repo root (the folder containing env_checker.py) and make it importable.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "env_checker.py").exists())
sys.path.insert(0, str(ROOT))

# Load .env into the environment for this session.
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ModuleNotFoundError:
    print("python-dotenv not installed yet — run: uv add python-dotenv")

# Verify .env variables and required packages.
from env_checker import run_checks
run_checks()

Environment variables (from .env.example)
  ✓ OPENAI_API_KEY  — set
  ✓ ANTHROPIC_API_KEY  — set
  ✓ LANGSMITH_TRACING  — set
  ✓ LANGSMITH_ENDPOINT  — set
  ✓ LANGSMITH_API_KEY  — set
  ✓ LANGSMITH_PROJECT  — set
  ✓ CHROMA_PERSIST_DIR  — set

Required packages (from pyproject.toml)
  ✓ beautifulsoup4  — installed (4.14.3)
  ✓ chromadb  — installed (1.5.9)
  ✓ langchain  — installed (1.3.2)
  ✓ langchain-chroma  — installed (1.1.0)
  ✓ langchain-community  — installed (0.4.2)
  ✓ langchain-core  — installed (1.4.0)
  ✓ langchain-openai  — installed (1.2.2)
  ✓ lxml  — installed (6.1.1)
  ✓ onnxruntime  — installed (1.19.2)
  ✓ pypdf  — installed (6.12.2)
  ✓ python-dotenv  — installed (1.2.2)
  ✓ ipykernel  — installed (7.2.0)
  ✓ jupyterlab  — installed (4.5.7)

✓ All checks passed.


True

# Recursive Chunking — Python (.py)

Source code shouldn't be split mid-statement. `RecursiveCharacterTextSplitter.from_language(Language.PYTHON)` seeds the splitter with **Python-specific separators** (`\nclass `, `\ndef `, `\n\tdef `, …) so chunk boundaries fall on classes and functions, keeping logical units intact.

Below we load a `.py` file and split it with language-aware recursive chunking.

## 1. Load the .py file

In [8]:
from langchain_community.document_loaders import PythonLoader

# Use a real source file from the repo as the sample.
sample = ROOT / "env_checker.py"
docs = PythonLoader(sample).load()

print(f"Loaded {len(docs)} document(s); {len(docs[0].page_content)} chars")

Loaded 1 document(s); 6053 chars


## 2. Language-aware recursive chunking

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language

# from_language() picks separators tuned for Python so splits land on
# code boundaries (classes/functions) rather than in the middle of a line.
splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size=400,      # characters per chunk
    chunk_overlap=50,    # characters shared between consecutive chunks
)

chunks = splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks")
print("Python separators used:", splitter._separators[:6])
print(f"\n--- first chunk ({len(chunks[0].page_content)} chars) ---")
print(chunks[0].page_content)

for i, chunk in enumerate(chunks[:6]):
    print(f"\n--- chunk {i+1} ({len(chunk.page_content)} chars) ---")
    print(chunk.page_content)

Split into 23 chunks
Python separators used: ['\nclass ', '\ndef ', '\n\tdef ', '\n\n', '\n', ' ']

--- first chunk (332 chars) ---
"""env_checker.py — sanity-check the rag-reference environment.

Verifies two things, both read dynamically from the project's own files:
  1. Every variable declared in .env.example is present and filled in (not a
     placeholder) in your local .env.
  2. Every package declared in pyproject.toml is installed in the current venv.

--- chunk 1 (332 chars) ---
"""env_checker.py — sanity-check the rag-reference environment.

Verifies two things, both read dynamically from the project's own files:
  1. Every variable declared in .env.example is present and filled in (not a
     placeholder) in your local .env.
  2. Every package declared in pyproject.toml is installed in the current venv.

--- chunk 2 (349 chars) ---
Nothing is hardcoded: add a key to .env.example or a dependency to
pyproject.toml and it is picked up automatically.

Usage:
    uv run python e